
### 과제
- 기간 : 7월 29일 (수)
- 09에서 진행한 과제에서 만든 코드를 FastAPI or express로 각각의 API 만들기
- 예스24_김도연.ipynb <- 앞에서 진행했던 파일에 그대로 붙여서 해도 상관없음


## FastAPI 서버 실행 방법

아래 코드 셀의 내용은 **`main.py`** 파일에 저장하여 실행해야 합니다.

### 실행 순서
1. 아래 셀을 실행하면 `main.py` 파일이 자동 생성됩니다.
2. 터미널에서 아래 명령어로 서버를 실행하세요:
   ```
   uvicorn main:app --reload
   ```
3. 브라우저에서 http://localhost:8000/docs 접속하면 Swagger UI에서 테스트 가능합니다.


In [1]:
# 이 셀을 실행하면 main.py 파일이 생성됩니다.
code = '''
import requests
from bs4 import BeautifulSoup
import time
import re
from fastapi import FastAPI, Query, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional

# --- Pydantic Models ---
class Book(BaseModel):
    rank: int = Field(..., description="베스트셀러 순위")
    title: str = Field(..., description="도서 제목")
    author: str = Field(..., description="저자")
    price: str = Field(..., description="판매가")
    sales_index: Optional[int] = Field(None, description="판매지수 (상세 페이지에서 수집)")
    detail_url: str = Field(..., description="도서 상세 정보 URL")

# --- FastAPI App ---
app = FastAPI(
    title="Yes24 Bestseller API",
    description="Yes24 종합 베스트셀러 목록을 크롤링하여 제공하는 API입니다.",
    version="1.0.0",
)

BASE_URL = "https://www.yes24.com"
BESTSELLER_URL = "/Product/Category/BestSeller"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}

def get_sales_index(detail_url: str) -> Optional[int]:
    try:
        res = requests.get(detail_url, headers=HEADERS, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        sell_num_tag = soup.select_one(".gd_sellNum")
        if not sell_num_tag:
            return None
        sell_text = sell_num_tag.get_text(strip=True)
        match = re.search(r"[\\d,]+", sell_text)
        if match:
            return int(match.group(0).replace(",", ""))
        return None
    except Exception:
        return None

@app.get("/", summary="API 상태 확인")
def root():
    return {"message": "Yes24 Bestseller API is running!", "docs": "/docs"}

@app.get("/bestsellers", response_model=List[Book], summary="Yes24 베스트셀러 목록 조회")
def get_bestsellers(
    pages: int = Query(1, ge=1, le=5, description="수집할 페이지 수 (1~5)"),
    include_sales_index: bool = Query(False, description="판매지수 포함 여부")
):
    bestsellers = []
    current_rank = 1

    for page in range(1, pages + 1):
        list_url = f"{BASE_URL}{BESTSELLER_URL}?categoryNumber=001&pageNumber={page}"
        try:
            res = requests.get(list_url, headers=HEADERS, timeout=10)
            res.raise_for_status()
            soup = BeautifulSoup(res.text, "html.parser")

            book_items = soup.select("#yesBestList > li:not(.ad)")
            if not book_items:
                break

            for item in book_items:
                title_tag = item.select_one(".gd_name")
                author_tag = item.select_one(".auth_pub")
                price_tag = item.select_one(".yes_b") or item.select_one(".sale_prc") or item.select_one(".price")

                if not title_tag:
                    continue

                title = title_tag.text.strip()
                detail_path = title_tag.get("href", "")
                detail_url = f"{BASE_URL}{detail_path}"
                author = author_tag.text.strip().split("|")[0].strip() if author_tag else "저자 정보 없음"
                price = price_tag.text.strip() if price_tag else "가격 정보 없음"

                sales_index = None
                if include_sales_index:
                    sales_index = get_sales_index(detail_url)
                    time.sleep(0.5)

                bestsellers.append(Book(
                    rank=current_rank,
                    title=title,
                    author=author,
                    price=price,
                    sales_index=sales_index,
                    detail_url=detail_url
                ))
                current_rank += 1

        except requests.RequestException as e:
            raise HTTPException(status_code=503, detail=f"페이지 {page} 크롤링 실패: {str(e)}")

        if page < pages:
            time.sleep(1)

    if not bestsellers:
        raise HTTPException(status_code=404, detail="베스트셀러 데이터를 가져올 수 없습니다.")

    return bestsellers

if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)
'''

with open('main.py', 'w', encoding='utf-8') as f:
    f.write(code.strip())

print('✅ main.py 파일이 생성되었습니다!')
print('📌 터미널에서 아래 명령어를 실행하세요:')
print('   uvicorn main:app --reload')
print('📌 또는 이 파일이 있는 폴더에서:')
print('   python main.py')

✅ main.py 파일이 생성되었습니다!
📌 터미널에서 아래 명령어를 실행하세요:
   uvicorn main:app --reload
📌 또는 이 파일이 있는 폴더에서:
   python main.py


In [ ]:
# FastAPI 코드 - main.py 내용 그대로 확인용

import requests
from bs4 import BeautifulSoup
import time
import re
from fastapi import FastAPI, Query, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional

# --- Pydantic Models for Data Structure ---
# API 응답의 데이터 구조를 정의합니다.
class Book(BaseModel):
    rank: int = Field(..., description="베스트셀러 순위")
    title: str = Field(..., description="도서 제목")
    author: str = Field(..., description="저자")
    price: str = Field(..., description="판매가")
    sales_index: Optional[int] = Field(None, description="판매지수 (상세 페이지에서 수집)")
    detail_url: str = Field(..., description="도서 상세 정보 URL")

# --- FastAPI App Initialization ---
app = FastAPI(
    title="Yes24 Bestseller API",
    description="Yes24 종합 베스트셀러 목록을 크롤링하여 제공하는 API입니다.",
    version="1.0.0",
)

# --- Helper Functions & Constants ---
BASE_URL = "https://www.yes24.com"
BESTSELLER_URL = "/Product/Category/BestSeller"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}

def get_sales_index(detail_url: str) -> Optional[int]:
    """
    상세 페이지로 이동하여 판매지수를 추출합니다.
    숫자만 추출하여 정수형으로 반환합니다.
    """
    try:
        res = requests.get(detail_url, headers=HEADERS, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        
        sell_num_tag = soup.select_one(".gd_sellNum")
        if not sell_num_tag:
            return None
            
        # '|판매지수 123,456판매지수란?' 형태의 텍스트에서 숫자만 추출
        sell_text = sell_num_tag.get_text(strip=True)
        match = re.search(r"[\d,]+", sell_text)
        if match:
            return int(match.group(0).replace(",", ""))
        return None
    except Exception:
        return None


# --- API Endpoints ---
@app.get("/", summary="API 상태 확인")
def root():
    return {"message": "Yes24 Bestseller API is running!", "docs": "/docs"}

@app.get("/bestsellers", response_model=List[Book], summary="Yes24 베스트셀러 목록 조회")
def get_bestsellers(
    pages: int = Query(1, ge=1, le=5, description="수집할 페이지 수 (1~5)"),
    include_sales_index: bool = Query(False, description="판매지수 포함 여부 (True시 요청 시간이 길어질 수 있음)")
):
    """
    Yes24 종합 베스트셀러 목록을 스크래핑하여 반환합니다.

    - **pages**: 가져올 베스트셀러 페이지 수를 지정합니다. (기본값: 1, 최대: 5)
    - **include_sales_index**: 각 도서의 상세 페이지에 방문하여 판매지수를 추가로 가져올지 여부.
      True로 설정하면 응답 시간이 크게 늘어날 수 있습니다.
    """
    bestsellers = []
    current_rank = 1

    for page in range(1, pages + 1):
        list_url = f"{BASE_URL}{BESTSELLER_URL}?categoryNumber=001&pageNumber={page}"
        
        try:
            res = requests.get(list_url, headers=HEADERS, timeout=10)
            res.raise_for_status()
            soup = BeautifulSoup(res.text, "html.parser")
            
            # yesBestList 내의 li 태그들을 찾되, 광고(class="ad")는 제외
            book_items = soup.select("#yesBestList > li:not(.ad)")

            if not book_items:
                break

            for item in book_items:
                # 기본 정보 추출
                title_tag = item.select_one(".gd_name")
                author_tag = item.select_one(".auth_pub")
                # 가격은 여러 셀렉터 시도 (예스24 구조 변경 대응)
                price_tag = item.select_one(".yes_b") or item.select_one(".sale_prc") or item.select_one(".price")

                # 제목이 없으면 건너뛰기
                if not title_tag:
                    continue

                title = title_tag.text.strip()
                detail_path = title_tag.get("href", "")
                detail_url = f"{BASE_URL}{detail_path}"
                author = author_tag.text.strip().split("|")[0].strip() if author_tag else "저자 정보 없음"
                price = price_tag.text.strip() if price_tag else "가격 정보 없음"

                # 판매지수 수집 (옵션)
                sales_index = None
                if include_sales_index:
                    sales_index = get_sales_index(detail_url)
                    time.sleep(0.5)  # 서버 부하 방지

                bestsellers.append(Book(
                    rank=current_rank,
                    title=title,
                    author=author,
                    price=price,
                    sales_index=sales_index,
                    detail_url=detail_url
                ))
                current_rank += 1

        except requests.RequestException as e:
            raise HTTPException(status_code=503, detail=f"페이지 {page} 크롤링 실패: {str(e)}")

        if page < pages:
            time.sleep(1)

    if not bestsellers:
        raise HTTPException(status_code=404, detail="베스트셀러 데이터를 가져올 수 없습니다.")

    return bestsellers

# uvicorn 직접 실행용
if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)

INFO:     Will watch for changes in these directories: ['d:\\project\\001-project\\web-dev-ai-2026\\07-python\\yes24-api']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [12428] using StatReload


## API 테스트

서버 실행 후 아래 셀로 API를 직접 테스트할 수 있습니다.

- 기본: `http://localhost:8000/bestsellers`
- 2페이지: `http://localhost:8000/bestsellers?pages=2`
- 판매지수 포함: `http://localhost:8000/bestsellers?pages=1&include_sales_index=true`
- Swagger UI: `http://localhost:8000/docs`


In [ ]:
# 서버 실행 중일 때 아래 코드로 API 테스트
import requests
import json

# 1페이지 베스트셀러 조회
response = requests.get("http://localhost:8000/bestsellers", params={"pages": 1})

if response.status_code == 200:
    books = response.json()
    print(f"✅ 총 {len(books)}권 조회 성공!")
    print()
    for book in books[:5]:  # 상위 5권만 출력
        print(f"#{book['rank']:2d} {book['title']}")
        print(f"     저자: {book['author']}")
        print(f"     가격: {book['price']}")
        print(f"     URL: {book['detail_url']}")
        print()
else:
    print(f"❌ 오류 발생: {response.status_code}")
    print(response.json())